# Task 3: Cross-Encoder JobBERT (Colab GPU)

**Runtime → Change runtime type → GPU (A100 / L4 / T4).** Semua logika ada di `task3_match/cross_encoder.py`; notebook ini hanya menjalankannya.

**Persiapan sekali saja**
1. Repo private → buat GitHub *fine-grained token* (Contents: read-only, hanya repo ini), simpan di Colab **Secrets** (ikon kunci di kiri) dengan nama `GH_TOKEN`, aktifkan *Notebook access*.
2. (Opsional) `HF_TOKEN` di Secrets agar unduhan Hugging Face lebih cepat.
3. Drive butuh ruang kosong ± 4 GB (8 model × ~420 MB) bila `COPY_MODELS = True`.

**Alur (8 run, sesuai TASKS.md)**
1. Tuning lr ∈ {2e-5, 3e-5} di **V1** untuk `cls` (3 kelas) dan `reg` (skor/100). Dipilih dengan **val** (Macro-F1 / MAE).
2. **V2** (+ entity Task 1) dan **V3** (+ profil Task 2) dengan lr terpilih: split, hyperparameter, seed sama.

**Penyimpanan & melanjutkan**
- Setelah tiap run, CSV metrik, prediksi, log, dan model disalin ke `MyDrive/task3_match/`. Run yang sudah ada di Drive **dilewati**, jadi jika Colab putus cukup jalankan ulang semua cell dari atas.
- Checkpoint per epoch disimpan di disk lokal Colab (`CKPT_ON_DRIVE = False`): run yang sedang berjalan saat putus diulang dari awal. Set `True` untuk menyimpan checkpoint ke Drive dan melanjutkan dari epoch terakhir, tetapi tiap checkpoint ±1,3 GB dan checkpoint lama yang dihapus masuk **Trash Drive** (tetap memakan kuota) sehingga perlu dikosongkan manual.

In [ ]:
BRANCH = "feat/task3-matching"
PROFILES = "profiles_t2.parquet"   # data/cache/: profil Task 2 (Field/Sub-field model Task 2, Level dari Task 1 seperti CLI)
COPY_MODELS = True
CKPT_ON_DRIVE = False

from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/task3_match"
CKPT = f"{OUT}/_ckpt" if CKPT_ON_DRIVE else "/content/ckpt"
!mkdir -p {OUT}/results {OUT}/models {OUT}/logs {CKPT}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
tok = userdata.get("GH_TOKEN")
REPO = "github.com/Ferdin-Arsenic/Job_Matching_System-NLP-.git"

def git(*args):
    """Jalankan git tanpa mencetak token: error disensor, dan remote disimpan tanpa token."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError("git gagal: " + (r.stderr or r.stdout).replace(tok, "***"))
    return r.stdout

%cd /content
if not os.path.exists("/content/repo"):
    git("clone", "-q", "-b", BRANCH, f"https://x-access-token:{tok}@{REPO}", "repo")
    git("-C", "repo", "remote", "set-url", "origin", f"https://{REPO}")   # token tidak tersimpan di .git/config
else:
    git("-C", "repo", "pull", "-q", f"https://x-access-token:{tok}@{REPO}", BRANCH)
%cd /content/repo
print(git("log", "--oneline", "-1"))
!pip install -q datasets accelerate
!ls data/splits data/cache outputs/ner

In [ ]:
import glob, time, pandas as pd

def tag_of(variant, kind, lr):
    return f"{variant}{'-stub' if variant == 'V3' and 'stub' in PROFILES else ''}_{kind}_lr{lr:g}"

def run(variant, kind, lr):
    tag = tag_of(variant, kind, lr)
    if os.path.exists(f"{OUT}/results/task3_ce_{tag}.csv"):
        print("skip (sudah ada di Drive):", tag); return
    t0 = time.time()
    cmd = (f"set -o pipefail; python task3_match/cross_encoder.py --variant {variant} --run {kind} --lr {lr} "
           f"--profiles {PROFILES} --ckpt-dir {CKPT} 2>&1 | tee -a {OUT}/logs/{tag}.log")
    subprocess.run(["bash", "-c", cmd], check=True)
    os.system(f"cp data/results/task3_ce_{tag}.csv data/results/task3_pred_{tag}.parquet {OUT}/results/")
    if COPY_MODELS: os.system(f"cp -r models/match/{tag} {OUT}/models/ && rm -rf models/match/{tag}")
    print(f"selesai {tag} dalam {(time.time() - t0) / 60:.1f} menit")

def results():
    return pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/results/task3_ce_*.csv")], ignore_index=True)

## 1. Tuning lr di V1 (4 run)

In [ ]:
for kind in ["cls", "reg"]:
    for lr in [2e-5, 3e-5]:
        run("V1", kind, lr)

In [ ]:
r = results().query("variant == 'V1' and split == 'val'")
best = {"cls": r.query("run == 'cls' and metric == 'macro_F1'").sort_values("value", ascending=False).lr.iloc[0],
        "reg": r.query("run == 'reg' and metric == 'MAE'").sort_values("value").lr.iloc[0]}
print("lr terpilih (val):", best)
r.query("metric in ['macro_F1', 'MAE', 'ROC_AUC', 'NDCG@5']").pivot_table(index=["run", "lr"], columns="metric", values="value").round(3)

## 2. Ablation V2, V3 dengan lr terpilih (4 run)

In [ ]:
for v in ["V2", "V3"]:
    for kind in ["cls", "reg"]:
        run(v, kind, best[kind])

## 3. Ringkasan test (seed 42) vs baseline

In [ ]:
cols = ["macro_F1", "ROC_AUC", "MAE", "Spearman", "Spearman_per_job", "NDCG@5", "MRR"]
ce = results().query("split == 'test'")
ce = ce[ce.apply(lambda x: x.lr == best[x.run], axis=1)]
base = pd.read_csv("data/results/task3_baselines.csv").query("split == 'test' and seed == 42")
table = pd.concat([base.pivot_table(index="model", columns="metric", values="value"),
                   ce.pivot_table(index="model", columns="metric", values="value")])[cols]
table.to_csv(f"{OUT}/results/task3_summary_seed42.csv")
table.round(3)

Selesai: unduh `MyDrive/task3_match/results/` ke `data/results/` di repo lokal (dan `models/` ke `models/match/` untuk CLI). Ingat: Macro-F1 bisa terbantu resume yang juga muncul di train (lihat baseline *Resume prior*); metrik per job (NDCG@5, MRR, Spearman_per_job) adalah ukuran matching yang lebih jujur.